# Coordinate frame transformation tests

In [ ]:
import math
import numpy as np
import pytest

from quantum_space_agi.scientific import (
    ScientificError,
    CoordinateFrame,
    RigidTransform,
    active_rotation_matrix_x,
    active_rotation_matrix_y,
    active_rotation_matrix_z,
    passive_rotation_matrix_x,
    euler_rotation_matrix,
    create_transform,
    transform_point,
    transform_vector,
    inverse_transform,
    compose_transforms,
)
from quantum_space_agi.scientific.units import get_unit


In [ ]:
def test_rotation_matrices_are_float64_and_orthogonal():
    for matrix in (active_rotation_matrix_x(0.2), active_rotation_matrix_y(-0.4), active_rotation_matrix_z(math.pi / 2)):
        assert matrix.dtype == np.float64
        assert np.allclose(matrix.T @ matrix, np.eye(3), atol=1e-12)
        assert np.isclose(np.linalg.det(matrix), 1.0, atol=1e-12)


def test_active_z_rotation_maps_x_axis_to_y_axis():
    matrix = active_rotation_matrix_z(math.pi / 2)
    assert np.allclose(matrix @ np.array([1.0, 0.0, 0.0]), np.array([0.0, 1.0, 0.0]), atol=1e-12)


def test_active_x_rotation_maps_y_axis_to_z_axis():
    matrix = active_rotation_matrix_x(math.pi / 2)
    assert np.allclose(matrix @ np.array([0.0, 1.0, 0.0]), np.array([0.0, 0.0, 1.0]), atol=1e-12)


def test_passive_rotation_is_transpose_of_active():
    active = active_rotation_matrix_x(0.37)
    passive = passive_rotation_matrix_x(0.37)
    assert np.allclose(passive, active.T, atol=1e-12)


def test_euler_fixed_axis_matches_explicit_matrix_product():
    angles = (0.1, -0.2, 0.3)
    result = euler_rotation_matrix("ZYX", angles, extrinsic=True)
    expected = active_rotation_matrix_z(angles[0]) @ active_rotation_matrix_y(angles[1]) @ active_rotation_matrix_x(angles[2])
    assert np.allclose(result, expected, atol=1e-12)


def test_rigid_point_round_trip():
    source = CoordinateFrame("body", unit=get_unit("m"), axes=("X", "Y", "Z"), handedness="right")
    target = CoordinateFrame("world", unit=get_unit("m"), axes=("X", "Y", "Z"), handedness="right")
    transform = create_transform(source, target, rotation=active_rotation_matrix_z(math.pi / 2), translation=(10.0, -2.0, 3.0))
    point = np.array([1.0, 2.0, 4.0], dtype=np.float64)
    mapped = transform_point(transform, point)
    restored = transform_point(inverse_transform(transform), mapped)
    assert mapped.dtype == np.float64
    assert np.allclose(restored, point, atol=1e-12)


def test_free_vector_ignores_translation():
    source = CoordinateFrame("A")
    target = CoordinateFrame("B")
    transform = create_transform(source, target, rotation=active_rotation_matrix_y(math.pi / 2), translation=(100.0, 200.0, 300.0))
    vector = (1.0, 2.0, 3.0)
    assert np.allclose(transform_vector(transform, vector), (3.0, 2.0, -1.0), atol=1e-12)


def test_transform_composition_and_round_trip():
    a = CoordinateFrame("A")
    b = CoordinateFrame("B")
    c = CoordinateFrame("C")
    ab = create_transform(a, b, rotation=active_rotation_matrix_z(math.pi / 2), translation=(1.0, 0.0, 0.0))
    bc = create_transform(b, c, rotation=active_rotation_matrix_x(math.pi / 2), translation=(0.0, 2.0, 0.0))
    point = np.array([2.0, 3.0, 4.0])
    direct = transform_point(bc, transform_point(ab, point))
    composed = compose_transforms(bc, ab)
    assert np.allclose(transform_point(composed, point), direct, atol=1e-12)
    assert np.allclose(transform_point(inverse_transform(composed), direct), point, atol=1e-12)


def test_unit_conversion_between_metre_and_kilometre_frames():
    metres = CoordinateFrame("METRES", unit=get_unit("m"))
    kilometres = CoordinateFrame("KILOMETRES", unit=get_unit("km"))
    transform = create_transform(metres, kilometres, translation=(0.0, 0.0, 0.0))
    mapped = transform_point(transform, (1000.0, 2000.0, 3000.0))
    assert np.allclose(mapped, (1.0, 2.0, 3.0), atol=1e-12)
    assert np.allclose(transform_point(inverse_transform(transform), mapped), (1000.0, 2000.0, 3000.0), atol=1e-12)

@pytest.mark.parametrize("values", [((1.0, 0.0, 0.0), (0.0, 0.0, 0.0)), ((1.0, 2.0),), ((True, False, True),), ((1.0, np.nan, 0.0),), ((1+0j, 0.0, 0.0),)])
def test_invalid_points_are_rejected(values):
    source = CoordinateFrame("A")
    target = CoordinateFrame("B")
    transform = create_transform(source, target)
    with pytest.raises(ScientificError):
        transform_point(transform, values)

@pytest.mark.parametrize("matrix", [np.eye(3) + np.diag([0.1, 0.0, 0.0]), np.array([[1.0, 0.0, 0.0],[0.0,-1.0,0.0],[0.0,0.0,1.0]])])
def test_invalid_rotation_matrices_are_rejected(matrix):
    with pytest.raises(ScientificError):
        create_transform(CoordinateFrame("A"), CoordinateFrame("B"), rotation=matrix)

@pytest.mark.parametrize("angle", [True, "90", np.nan, np.inf])
def test_invalid_rotation_angles_are_rejected(angle):
    with pytest.raises(ScientificError):
        active_rotation_matrix_z(angle)

@pytest.mark.parametrize("sequence, angles", [("XXY", (0.1, 0.2, 0.3)), ("XYZ", (0.1, 0.2)), ("XYZ", (True, 0.2, 0.3))])
def test_invalid_euler_inputs_are_rejected(sequence, angles):
    with pytest.raises(ScientificError):
        euler_rotation_matrix(sequence, angles)

def test_composition_rejects_disconnected_frames():
    ab = create_transform(CoordinateFrame("A"), CoordinateFrame("B"))
    cd = create_transform(CoordinateFrame("C"), CoordinateFrame("D"))
    with pytest.raises(ScientificError) as exc:
        compose_transforms(ab, cd)
    assert exc.value.code.value == "FRAME_MISMATCH"

def test_left_handed_frame_is_rejected():
    source = CoordinateFrame("A", handedness="left")
    with pytest.raises(ScientificError):
        create_transform(source, CoordinateFrame("B"))


In [ ]:
# arc04-test-runner
# Notebook cases are discovered and expanded by tests/test_notebooks.py.
